In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import importlib
import matplotlib.pyplot as plt

import env.trading_env as te
import features.feature_engineering as fe
import backtest.backtest as bt
from stable_baselines3 import PPO, SAC, A2C

importlib.reload(te)
importlib.reload(fe)
importlib.reload(bt)

<module 'backtest.backtest' from 'C:\\DKU\\26spring\\stats402\\project\\notebooks\\..\\backtest\\backtest.py'>

In [2]:
# Reading data
df = pd.read_excel('../data/data.xlsx', skiprows=6, header=0)
df.columns = ['date', 'gold', 'silver', 'copper']
df = df[pd.to_datetime(df['date'], errors='coerce').notna()]
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Feature Engineering
features_final = fe.build_features(df)

# Split
train = features_final[features_final.index <= '2023-12-31']
test = features_final[features_final.index >= '2024-01-01']
price_train = df[df.index.isin(train.index)]
price_test = df[df.index.isin(test.index)]

print(f"Train: {len(train)}days, Test: {len(test)}days")

Train: 2592days, Test: 485days


In [3]:
# Load model
ppo = PPO.load("../agents/ppo_step_1m.zip")
sac = SAC.load("../agents/sac_step_1m.zip")
a2c = A2C.load("../agents/a2c_step_1m.zip")

# Initialise the test environment
test_env_ppo = te.MetalTradingEnv(test, price_test)
test_env_sac = te.MetalTradingEnv(test, price_test)
test_env_a2c = te.MetalTradingEnv(test, price_test)
test_env_ew = te.MetalTradingEnv(test, price_test)
test_env_bh = te.MetalTradingEnv(test, price_test)

# Run the backtest
print("Running PPO...")
ppo_results = bt.run_agent(ppo, test_env_ppo)

print("Running SAC...")
sac_results = bt.run_agent(sac, test_env_sac)

print("Running A2C...")
a2c_results = bt.run_agent(a2c, test_env_a2c)

print("Running Equal Weight...")
ew_values = bt.run_equal_weight(test_env_ew)

print("Running Buy and Hold...")
bh_values = bt.run_buy_and_hold(test_env_bh)

print("Done!")

Running PPO...
Running SAC...
Running A2C...
Running Equal Weight...
Running Buy and Hold...
Done!


In [4]:
# Calculate the metrics for each strategy
strategies = {
    'PPO': ppo_results['portfolio_values'],
    'SAC': sac_results['portfolio_values'],
    'A2C': a2c_results['portfolio_values'],
    'Equal Weight': ew_values,
    'Buy and Hold': bh_values
}

# Summary
metrics_dict = {}
for name, values in strategies.items():
    metrics_dict[name] = bt.compute_metrics(values)

metrics_df = pd.DataFrame(metrics_dict).T
metrics_df.columns = ['Cumulative Return', 'Annualized Return', 'Annualized Vol', 'Sharpe', 'Max Drawdown']

# Formatted display
metrics_display = metrics_df.copy()
metrics_display['Cumulative Return'] = metrics_df['Cumulative Return'].map('{:.2%}'.format)
metrics_display['Annualized Return'] = metrics_df['Annualized Return'].map('{:.2%}'.format)
metrics_display['Annualized Vol'] = metrics_df['Annualized Vol'].map('{:.2%}'.format)
metrics_display['Sharpe'] = metrics_df['Sharpe'].map('{:.4f}'.format)
metrics_display['Max Drawdown'] = metrics_df['Max Drawdown'].map('{:.2%}'.format)

print(metrics_display)

             Cumulative Return Annualized Return Annualized Vol  Sharpe  \
PPO                     62.54%            28.78%         14.44%  1.9925   
SAC                     18.93%             9.45%         13.79%  0.6854   
A2C                     58.00%            26.89%         10.07%  2.6700   
Equal Weight           110.90%            47.48%         14.55%  3.2639   
Buy and Hold            58.49%            27.09%         13.45%  2.0151   

             Max Drawdown  
PPO                -8.60%  
SAC               -23.15%  
A2C                -6.54%  
Equal Weight      -12.88%  
Buy and Hold      -13.40%  


In [5]:
import os
files = os.listdir('../agents/')
print([f for f in files if 'v2' in f])

['a2c_v2_1m.zip', 'a2c_v2_rewards_1m.npy', 'ppo_v2_1m.zip', 'ppo_v2_rewards_1m.npy', 'sac_v2_1m.zip', 'sac_v2_rewards_1m.npy']


In [6]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import importlib
import matplotlib.pyplot as plt

import env.trading_env_v2 as te2
import features.feature_engineering as fe
import backtest.backtest as bt

from stable_baselines3 import PPO, SAC, A2C

importlib.reload(te2)
importlib.reload(fe)
importlib.reload(bt)


df = pd.read_excel('../data/data.xlsx', skiprows=6, header=0)
df.columns = ['date', 'gold', 'silver', 'copper']
df = df[pd.to_datetime(df['date'], errors='coerce').notna()]
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

features_final = fe.build_features(df)

train = features_final[features_final.index <= '2023-12-31']
test = features_final[features_final.index >= '2024-01-01']
price_train = df[df.index.isin(train.index)]
price_test = df[df.index.isin(test.index)]

print(f"Train: {len(train)}days, Test: {len(test)}days")

Train: 2592days, Test: 485days


In [8]:
# Load the v2 model
ppo_v2 = PPO.load("../agents/ppo_v2_1m.zip")

a2c_v2 = A2C.load("../agents/a2c_v2_1m.zip")

# Initialise the test environment
test_env_ppo = te2.MetalTradingEnvV2(test, price_test)

test_env_a2c = te2.MetalTradingEnvV2(test, price_test)
test_env_ew = te2.MetalTradingEnvV2(test, price_test)
test_env_bh = te2.MetalTradingEnvV2(test, price_test)

# Run the backtest
print("Running PPO...")
ppo_results = bt.run_agent(ppo_v2, test_env_ppo)


print("Running A2C...")
a2c_results = bt.run_agent(a2c_v2, test_env_a2c)

print("Running Equal Weight...")
ew_values = bt.run_equal_weight(test_env_ew)

print("Running Buy and Hold...")
bh_values = bt.run_buy_and_hold(test_env_bh)

print("Done!")

Running PPO...
Running A2C...
Running Equal Weight...
Running Buy and Hold...
Done!


In [9]:

strategies = {
    'PPO V2': ppo_results['portfolio_values'],
    'A2C V2': a2c_results['portfolio_values'],
    'Equal Weight': ew_values,
    'Buy and Hold': bh_values
}

metrics_dict = {}
for name, values in strategies.items():
    metrics_dict[name] = bt.compute_metrics(values)

metrics_df = pd.DataFrame(metrics_dict).T
metrics_df.columns = ['Cumulative Return', 'Annualized Return', 'Annualized Vol', 'Sharpe', 'Max Drawdown']

metrics_display = metrics_df.copy()
metrics_display['Cumulative Return'] = metrics_df['Cumulative Return'].map('{:.2%}'.format)
metrics_display['Annualized Return'] = metrics_df['Annualized Return'].map('{:.2%}'.format)
metrics_display['Annualized Vol'] = metrics_df['Annualized Vol'].map('{:.2%}'.format)
metrics_display['Sharpe'] = metrics_df['Sharpe'].map('{:.4f}'.format)
metrics_display['Max Drawdown'] = metrics_df['Max Drawdown'].map('{:.2%}'.format)

print(metrics_display)

             Cumulative Return Annualized Return Annualized Vol  Sharpe  \
PPO V2                  20.27%            10.09%         10.83%  0.9319   
A2C V2                 122.36%            51.60%         16.08%  3.2082   
Equal Weight           108.78%            46.71%         14.54%  3.2115   
Buy and Hold            58.86%            27.25%         13.12%  2.0778   

             Max Drawdown  
PPO V2            -15.54%  
A2C V2            -10.62%  
Equal Weight      -12.98%  
Buy and Hold      -14.15%  


In [13]:
import env.trading_env_v3 as te3
importlib.reload(te3)

ppo_v3 = PPO.load("../agents/ppo_v3_1.2m")
a2c_v3 = A2C.load("../agents/a2c_v3_1.4m")

test_env_ppo_v3 = te3.MetalTradingEnvV3(test, price_test)
test_env_a2c_v3 = te3.MetalTradingEnvV3(test, price_test)

print("Running PPO V3...")
ppo_v3_results = bt.run_agent(ppo_v3, test_env_ppo_v3)

print("Running A2C V3...")
a2c_v3_results = bt.run_agent(a2c_v3, test_env_a2c_v3)

print("Done!")

Running PPO V3...
Running A2C V3...
Done!


In [14]:
strategies_all = {
    'PPO V2': ppo_results['portfolio_values'],
    'A2C V2': a2c_results['portfolio_values'],
    'PPO V3': ppo_v3_results['portfolio_values'],
    'A2C V3': a2c_v3_results['portfolio_values'],
    'Equal Weight': ew_values,
    'Buy and Hold': bh_values
}

metrics_dict = {}
for name, values in strategies_all.items():
    metrics_dict[name] = bt.compute_metrics(values)

metrics_df = pd.DataFrame(metrics_dict).T
metrics_df.columns = ['Cumulative Return', 'Annualized Return', 'Annualized Vol', 'Sharpe', 'Max Drawdown']

metrics_display = metrics_df.copy()
metrics_display['Cumulative Return'] = metrics_df['Cumulative Return'].map('{:.2%}'.format)
metrics_display['Annualized Return'] = metrics_df['Annualized Return'].map('{:.2%}'.format)
metrics_display['Annualized Vol'] = metrics_df['Annualized Vol'].map('{:.2%}'.format)
metrics_display['Sharpe'] = metrics_df['Sharpe'].map('{:.4f}'.format)
metrics_display['Max Drawdown'] = metrics_df['Max Drawdown'].map('{:.2%}'.format)

print(metrics_display)

             Cumulative Return Annualized Return Annualized Vol  Sharpe  \
PPO V2                  20.27%            10.09%         10.83%  0.9319   
A2C V2                 122.36%            51.60%         16.08%  3.2082   
PPO V3                  26.59%            13.06%         12.94%  1.0090   
A2C V3                  15.83%             7.95%          7.51%  1.0583   
Equal Weight           108.78%            46.71%         14.54%  3.2115   
Buy and Hold            58.86%            27.25%         13.12%  2.0778   

             Max Drawdown  
PPO V2            -15.54%  
A2C V2            -10.62%  
PPO V3            -11.06%  
A2C V3            -10.13%  
Equal Weight      -12.98%  
Buy and Hold      -14.15%  
